# 26 — PydanticAI Agent

**Pattern:** Schema-first agent framework -- define the output type first, the framework handles the rest.  
**Contrast with:** LangChain `with_structured_output()` (example 3).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/26-pydantic-ai-agent/pydantic_ai_agent_workbook.ipynb)

In [ ]:
%pip install -q pydantic-ai python-dotenv

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## 1. Schema-first design

LangChain: start with the chain, attach a schema -- `llm.with_structured_output(Schema)`  
PydanticAI: start with the schema, build the agent -- `Agent(model, result_type=Schema)`

PydanticAI treats the output type as the primary artifact. Validation, retry, and prompt injection all derive from it.

## 2. Schema

In [ ]:
from pydantic import BaseModel, Field


class LineItem(BaseModel):
    description: str = Field(description="Description of the goods or service.")
    quantity: float = Field(description="Number of units.")
    unit_price: float = Field(description="Price per unit.")
    total: float = Field(description="Line total (quantity x unit_price).")


class Invoice(BaseModel):
    vendor: str = Field(description="Name of the vendor.")
    invoice_number: str = Field(description="Invoice reference number.")
    invoice_date: str = Field(description="Date issued (YYYY-MM-DD if parseable).")
    due_date: str = Field(description="Payment due date.")
    currency: str = Field(description="Three-letter ISO currency code.")
    subtotal: float = Field(description="Sum before tax.")
    tax: float = Field(description="Total tax. Use 0.0 if not stated.")
    total_due: float = Field(description="Total amount due including tax.")
    line_items: list[LineItem] = Field(description="Line items on the invoice.")

## 3. Agent

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel

model = OpenAIModel("gpt-4o-mini", api_key=os.environ["OPENAI_API_KEY"])

agent = Agent(
    model,
    result_type=Invoice,
    system_prompt=(
        "You are an invoice parsing assistant. "
        "Extract all fields exactly as stated. "
        "Use 0.0 for missing numeric fields and empty string for missing text fields."
    ),
)


def extract(invoice_text: str) -> Invoice:
    result = agent.run_sync(f"Extract the invoice fields:\n\n{invoice_text}")
    return result.data

## 4. Run

In [ ]:
sample = """INVOICE
Vendor: Acme Software Inc.
Invoice #: INV-2024-0891  Date: 2024-11-01  Due: 2024-11-30  Currency: USD

Professional Plan (monthly)  1 x $299.00  $299.00
Additional seats (5)         5 x $49.00   $245.00

Subtotal: $544.00  Tax (8%): $43.52  TOTAL DUE: $587.52"""

inv = extract(sample)
print(f"Vendor:  {inv.vendor}")
print(f"Invoice: {inv.invoice_number}")
print(f"Total:   {inv.currency} {inv.total_due}")
for item in inv.line_items:
    print(f"  {item.description}: {item.quantity} x {item.unit_price} = {item.total}")

## 5. Starter Exercise

Add a `payment_terms` field to `Invoice` (e.g. `"Net 30"`).  
PydanticAI will automatically include it in the extraction prompt -- no other change needed.

In [ ]:
# Your code here

### Answer Key

In [ ]:
class InvoiceV2(BaseModel):
    vendor: str
    invoice_number: str
    invoice_date: str
    due_date: str
    payment_terms: str = Field(default="", description="Payment terms (e.g. Net 30).")
    currency: str
    subtotal: float
    tax: float
    total_due: float
    line_items: list[LineItem]


agent_v2 = Agent(
    model,
    result_type=InvoiceV2,
    system_prompt="You are an invoice parsing assistant. Extract all fields exactly as stated.",
)
r = agent_v2.run_sync(f"Extract the invoice fields:\n\n{sample}")
print(f"Payment terms: {r.data.payment_terms or '(not stated)'}")